# Whisper-Medium LoRA Fine-tuning for Maltese ASR — Retrain v2

**This is the retrain of the April Whisper-Medium notebook with a proper train/val/test split and the methodological fixes identified in the supervision review.**

## Changes from the previous version

1. **Proper train/val/test split.** The 4,481-utterance train CSV is split 90/10 into train (~4,033) and val (~448); the 498-utterance test CSV is held out. The val set is used for early-stopping and best-checkpoint selection during training. The test set is touched exactly twice — once for the zero-shot baseline and once for the final evaluation — and neither call informs any model-selection decision. This follows the protocol used by Mainzinger & Levow (2024, ACL SRW) for low-resource MMS adapter fine-tuning and matches the train/val/test partitioning used by Williams, DeMarco & Borg (2023, SIGUL) on the original MASRI corpus.

2. **New save directory.** Models save to `Whisper_Medium_LoRA_Maltese_v2_proper_split` so the original checkpoint is preserved.

3. **Zero-shot baseline added.** Evaluates the base Whisper-Medium model on the test set before any LoRA training, so the fine-tuning contribution is quantifiable.

## Hyperparameters (defended in thesis methodology)

- **LR = 1e-3, warmup = 50:** canonical HuggingFace PEFT Whisper-LoRA configuration (Mangrulkar & Paul, 2023). Independently confirmed in Song et al. (2024, Interspeech) "LoRA-Whisper", which also establishes r=32 as the optimal LoRA rank for Whisper. Previous overfitting at this LR was a symptom of the test-set contamination in the previous run; with a proper val set, early-stopping on val WER catches the generalisation peak.
- **LoRA r=32, α=64, q/v targets, dropout=0.05:** held identical across all 3 models for fair architectural comparison. r=32 ablation from Song et al. (2024) shows no further gain at higher ranks.
- **Beam search width=5 at evaluation:** matched between the two seq2seq models (Whisper, Seamless) so generation quality is directly comparable.
- **Early stopping patience=3 on val WER**, seed=42, identical text normalisation, FP16, gradient checkpointing.


## 1. Setup

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate jiwer librosa soundfile sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.3 MB/s eta 0:00:00


In [ ]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import torch
from transformers import set_seed

# Mitigate CUDA memory fragmentation on T4 — relevant during beam-search eval
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Drive
drive.mount('/content/drive')

# Reproducibility — identical seed across all 3 model notebooks
set_seed(42)

# Paths
model_id = "openai/whisper-medium"
model_save_dir = "/content/drive/My Drive/ASRModels/Whisper_Medium_LoRA_Maltese_v2_proper_split"
train_csv_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/train_metadata.csv"
test_csv_path  = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/test_metadata.csv"

os.makedirs(model_save_dir, exist_ok=True)
print(f"Models will be saved to: {model_save_dir}")

Mounted at /content/drive
Models will be saved to: /content/drive/My Drive/ASRModels/Whisper_Medium_LoRA_Maltese_v2_proper_split


## 2. Data preparation

Loads MASRI Headset v2, performs the 90/10 train/val split, fixes Windows paths, casts audio to 16 kHz, and applies the text normalisation used at scoring.

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
import re

# Load all CSVs. The original train CSV becomes train+val; the test CSV is held out.
raw = load_dataset("csv", data_files={
    "trainval": train_csv_path,
    "test":     test_csv_path,
})

# Fix Windows paths -> Colab paths
drive_base_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/"

def fix_paths_bulletproof(batch):
    old_path = batch["file_path"].replace("\\", "/")
    if "/speech/" in old_path:
        relative_path = "speech/" + old_path.split("/speech/")[-1]
    else:
        relative_path = old_path.split("/")[-1]
    batch["file_path"] = os.path.join(drive_base_path, relative_path)
    return batch

raw = raw.map(fix_paths_bulletproof)

# 90/10 train/val split of the trainval portion (Mainzinger & Levow 2024 protocol).
# Seed matches the global seed so the split is deterministic across the 3 notebooks.
split = raw["trainval"].train_test_split(test_size=0.10, seed=42, shuffle=True)

masri_dataset = DatasetDict({
    "train": split["train"],
    "val":   split["test"],     # 10% carved out of trainval — used for early stopping
    "test":  raw["test"],       # held-out test set — touched only at the very end
})

print(f"Train samples: {len(masri_dataset['train'])}")
print(f"Val   samples: {len(masri_dataset['val'])}")
print(f"Test  samples: {len(masri_dataset['test'])}  (held out — touched only at zero-shot baseline and final evaluation)")

# Text normalisation — IDENTICAL across all 3 model notebooks for fair comparison
def clean_text(batch):
    text = batch["transcription"].lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    batch["transcription"] = re.sub(r"\s+", " ", text).strip()
    return batch

masri_dataset = masri_dataset.map(clean_text)

# Cast audio column to 16 kHz
masri_dataset = masri_dataset.cast_column("file_path", Audio(sampling_rate=16000))


Generating trainval split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4481 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Train samples: 4032
Val   samples: 449
Test  samples: 498  (held out — touched only at zero-shot baseline and final evaluation)


Map:   0%|          | 0/4032 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

# Whisper's multilingual processor — language and task baked in
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="Maltese", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="Maltese", task="transcribe")

print("Whisper feature extractor, tokenizer, and processor loaded.")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Whisper feature extractor, tokenizer, and processor loaded.


In [ ]:
def prepare_dataset(batch):
    audio = batch["file_path"]
    # Compute 80-channel log-Mel features from 16 kHz audio (Whisper's expected input)
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # Tokenise transcription using Whisper's BPE tokenizer
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

masri_dataset = masri_dataset.map(
    prepare_dataset,
    remove_columns=masri_dataset.column_names["train"],
    num_proc=8,
)

print("Dataset preprocessing complete.")

Map (num_proc=8):   0%|          | 0/4032 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/449 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/498 [00:00<?, ? examples/s]

Dataset preprocessing complete.


## 3. Data collator and metrics

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from jiwer import wer as compute_wer, cer as compute_cer
import numpy as np

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids":      f["labels"]}         for f in features]

        # Pad audio inputs (returns zero-padded log-Mel features)
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad text labels
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 so cross-entropy loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip leading BOS if tokenizer added it — generate() will prepend it again
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Same normaliser as training-time, applied to BOTH preds and refs at scoring
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 in labels so we can decode
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Filter empty references (jiwer divides by ref word count)
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 1.0, "cer": 1.0}
    pred_str, label_str = zip(*pairs)

    return {
        "wer": compute_wer(list(label_str), list(pred_str)),
        "cer": compute_cer(list(label_str), list(pred_str)),
    }

## 4. Model and LoRA configuration

LoRA on q/v projections only (no `modules_to_save`) — Whisper-Medium's decoder head was already trained on Maltese as part of its multilingual pretraining (96 languages), so the LM head is left frozen and adapted via LoRA only. `enable_input_require_grads()` is required when combining `gradient_checkpointing=True` with PEFT to ensure gradients flow through the LoRA adapters.

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model
from functools import partial

# Load base model
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Disable cache (incompatible with gradient_checkpointing during training)
model.config.use_cache = False

# Bind language and task to generate() so eval calls produce Maltese transcripts
model.generate = partial(model.generate, language="Maltese", task="transcribe")

# REQUIRED with gradient_checkpointing + PEFT — gradients won't flow without this
model.enable_input_require_grads()

# LoRA config — IDENTICAL to the other 2 model notebooks for fair comparison
config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    # No modules_to_save: Whisper's decoder head is already Maltese-aware
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204


In [ ]:
# Clear legacy generation attributes from main config (these conflict with trainer)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None
model.config.use_cache = False

# Set generation behaviour in the dedicated generation_config
model.generation_config.language = "maltese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

## 5. Trainer setup

`eval_dataset` is the VAL split — early stopping and best-checkpoint selection use val WER, never test WER. The trainer is constructed once and reused for the zero-shot baseline, training, and the final test evaluation.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=model_save_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,             # smaller for beam-search eval; prevents T4 OOM
    gradient_accumulation_steps=4,            # effective batch size 32 (matched across all 3 models)
    eval_accumulation_steps=8,                # offload eval preds to CPU every 8 batches (memory)
    learning_rate=1e-3,                       # canonical Whisper LoRA LR (Mangrulkar & Paul 2023; Song et al. 2024)
    warmup_steps=50,                          # canonical Whisper LoRA warmup
    num_train_epochs=15,                      # ceiling — early stopping decides actual stop
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=5,                   # matched with SeamlessM4T for fair seq2seq eval
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",              # evaluated on the VAL split, not test
    greater_is_better=False,
    remove_unused_columns=False,              # required for PEFT
    gradient_checkpointing=True,              # paired with enable_input_require_grads() above
    seed=42,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=masri_dataset["train"],
    eval_dataset=masri_dataset["val"],        # VAL, not test — see Mainzinger & Levow 2024
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## 6. Zero-shot baseline on the test set

Single call on the test set with the base Whisper-Medium model (LoRA adapters are zero-initialised so the wrapped model behaves like the base). This number is reported as the pre-fine-tuning baseline; it does not inform any training decision.

In [ ]:
print("=== Zero-shot baseline (Whisper-Medium, no fine-tuning) on TEST set ===")
zero_shot_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="zero_shot",
)
for k, v in zero_shot_metrics.items():
    print(f"  {k}: {v}")

# Save zero-shot baseline to disk
import json as _json
with open(os.path.join(model_save_dir, "zero_shot_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in zero_shot_metrics.items()}, f, indent=2)

=== Zero-shot baseline (Whisper-Medium, no fine-tuning) on TEST set ===


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


  zero_shot_loss: 4.649367332458496
  zero_shot_model_preparation_time: 0.1787
  zero_shot_wer: 0.9595862861520782
  zero_shot_cer: 0.38090810475747716
  zero_shot_runtime: 1483.4441
  zero_shot_samples_per_second: 0.336
  zero_shot_steps_per_second: 0.168


## 7. Training

In [ ]:
print("Starting Whisper-Medium LoRA fine-tuning (early stopping on val WER)")
trainer.train()

Starting Whisper-Medium LoRA fine-tuning (early stopping on val WER)


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer,Cer
1,3.085225,0.598830,0.178700,0.447851,0.126614
2,1.750466,0.501906,0.178700,0.362537,0.104770
3,1.140191,0.460556,0.178700,0.330204,0.091925
4,0.819759,0.457065,0.178700,0.329787,0.094066
5,0.546568,0.449408,0.178700,0.296621,0.090520
6,0.378804,0.463941,0.178700,0.308302,0.084398
7,0.272255,0.461778,0.178700,0.294118,0.080953
8,0.189619,0.474965,0.178700,0.305590,0.085636
9,0.119268,0.484166,0.178700,0.296204,0.081254
10,0.089992,0.480628,0.178700,0.288903,0.080886


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer,Cer
1,3.085225,0.598830,0.178700,0.447851,0.126614
2,1.750466,0.501906,0.178700,0.362537,0.104770
3,1.140191,0.460556,0.178700,0.330204,0.091925
4,0.819759,0.457065,0.178700,0.329787,0.094066
5,0.546568,0.449408,0.178700,0.296621,0.090520
6,0.378804,0.463941,0.178700,0.308302,0.084398
7,0.272255,0.461778,0.178700,0.294118,0.080953
8,0.189619,0.474965,0.178700,0.305590,0.085636
9,0.119268,0.484166,0.178700,0.296204,0.081254
10,0.089992,0.480628,0.178700,0.288903,0.080886


TrainOutput(global_step=1890, training_loss=0.7083563898290907, metrics={'train_runtime': 55976.7973, 'train_samples_per_second': 1.08, 'train_steps_per_second': 0.034, 'total_flos': 6.25480804859904e+19, 'train_loss': 0.7083563898290907, 'epoch': 15.0})

## 8. Save and final test-set evaluation

In [ ]:
# Save best model (loaded automatically thanks to load_best_model_at_end)
trainer.save_model(model_save_dir)
processor.save_pretrained(model_save_dir)
print(f"Model saved to {model_save_dir}")

# FINAL evaluation on the held-out test set — touched once, after all training decisions are locked in
print("\n=== FINAL test-set evaluation (held-out, untouched during training) ===")
final_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="final_test",
)
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

# Save final metrics to disk for the comparison table
with open(os.path.join(model_save_dir, "final_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in final_metrics.items()}, f, indent=2)
print(f"\nMetrics saved to {model_save_dir}/final_test_metrics.json")

Model saved to /content/drive/My Drive/ASRModels/Whisper_Medium_LoRA_Maltese_v2_proper_split

=== FINAL test-set evaluation (held-out, untouched during training) ===


early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


  final_test_loss: 0.43166521191596985
  final_test_model_preparation_time: 0.1787
  final_test_wer: 0.2509097873970504
  final_test_cer: 0.07050984038431737
  final_test_runtime: 1579.6132
  final_test_samples_per_second: 0.315
  final_test_steps_per_second: 0.158
  epoch: 15.0

Metrics saved to /content/drive/My Drive/ASRModels/Whisper_Medium_LoRA_Maltese_v2_proper_split/final_test_metrics.json
